# Ce que vous saurez fabriquer en trois clics · *What you'll be able to make in three clicks*

Notebook compagnon du chapitre **18. Atelier données : découvrir FRED, l'entrepôt de la Réserve fédérale** — [lire l'article](https://nmlab.io/ressources/atelier-donnees-decouvrir-fred).
Companion notebook to chapter **18. Data Workshop: Discovering FRED, the Federal Reserve's Warehouse** — [read the article](https://nmlab.io/en/ressources/data-workshop-discovering-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_data() -> tuple[Series, Series]:
    """Taux de chômage (UNRATE) et indicateur de récession NBER (USREC), en direct de FRED.
    Unemployment rate and NBER recession flag, live from FRED."""
    unemployment = nm.load_fred("UNRATE")
    recessions = nm.load_fred("USREC", start=str(unemployment.index[0].year))
    return unemployment, recessions

unemployment, recessions = load_data()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Ce que vous saurez fabriquer en trois clics",
        sub="Taux de chômage américain, avec les récessions du NBER — un graphique FRED type",
        ylab="taux de chômage, %",
        bands="bandes grises =\nrécessions (NBER)",
        note="Une série, une case « bandes de récession » à cocher, et l'histoire du cycle apparaît. Chaque pic de chômage\n"
             "épouse une bande grise. Source : BLS et NBER via FRED (UNRATE, USREC)."),
    "en": dict(
        title="What you'll be able to make in three clicks",
        sub="U.S. unemployment rate, with NBER recessions — a typical FRED chart",
        ylab="unemployment rate, %",
        bands="grey bands =\nrecessions (NBER)",
        note="One series, one « recession bars » box to tick, and the history of the cycle appears. Every unemployment\n"
             "peak hugs a grey band. Source: BLS and NBER via FRED (UNRATE, USREC)."),
}

def build_figure(unemployment: Series, recessions: Series, lang: str) -> Figure:
    """Chômage sur toute la période, ombré des récessions du NBER."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig)

    # Ombrer chaque période de récession (suites contiguës où USREC == 1).
    # Shade each recession (contiguous runs where USREC == 1).
    runs = recessions.ne(recessions.shift()).cumsum()
    for _, run in recessions.groupby(runs):
        if run.iloc[0] == 1:
            ax.axvspan(run.index[0], run.index[-1], color=nm.COLORS["edge"], alpha=0.75, linewidth=0)

    ax.plot(unemployment.index, unemployment, color=nm.COLORS["blue"], linewidth=2.9)
    ax.set_ylim(0, 15.5)
    ax.set_yticks(range(0, 15, 2))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.012)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.text(0.085, 0.70, text["bands"], transform=ax.transAxes, fontsize=21.5,
            color=nm.COLORS["muted"], linespacing=1.55)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(unemployment, recessions, LANG)